In [ ]:
import pandas as pd
import numpy as np
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score, confusion_matrix
import time
import re
import warnings
warnings.filterwarnings('ignore')

performance = pd.DataFrame(columns=['Name', 'Accuracy', 'Precision', 'Sensitivity', 'F1 Score', 'MCC', 'Markedness', "Youden's J", 'FMI', 'Time'])

df = pd.read_csv("StealthPhisher2025.csv")
LABEL = df.iloc[:,-1:].columns[0]

colsAll = df.select_dtypes(include=['float64','int64']).columns
colsAll.append(pd.Index([LABEL]))
df = pd.DataFrame(df[colsAll]).copy()

print('Dataset Shape: ', df.shape)
df.iloc[:,-1:].value_counts()

X = df.drop(columns=[LABEL]).values
y = df[LABEL].values

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
import pandas as pd
import numpy as np
import time
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input, Embedding, Flatten, concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, roc_auc_score,
                             confusion_matrix)

# Assuming `df` is your DataFrame and `LABEL` is the target column
X = df.drop(columns=[LABEL]).values
y = df[LABEL].values

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Define the Wide and Deep Neural Network
def build_wide_and_deep(input_dim):
    # Input layer
    inputs = Input(shape=(input_dim,))
    wide = Dense(1)(inputs)
    deep = Dense(128, activation='relu')(inputs)
    deep = Dense(64, activation='relu')(deep)
    deep = Dense(32, activation='relu')(deep)
    combined = concatenate([wide, deep])
    outputs = Dense(1, activation='sigmoid')(combined)    
    model = Model(inputs=inputs, outputs=outputs)
    return model

model = build_wide_and_deep(input_dim=X_train.shape[1])
model.compile(optimizer=Adam(learning_rate=0.01), loss='binary_crossentropy', metrics=['accuracy'])
start_time = time.time()
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=1)
end_time = time.time()
y_pred = (model.predict(X_test) > 0.5).astype("int32")
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
markedness = (precision * (tp / (tp + fn))) - ((1 - precision) * (fn / (fn + tp))) if (tp + fn) > 0 and (fn + tp) > 0 else np.nan
youdens_j = sensitivity + (tn / (tn + fp)) - 1
fmi = np.sqrt(precision * sensitivity)

total_time = end_time - start_time

newRow = {
    'Name': 'Wide and Deep Neural Network',
    'Accuracy': accuracy,
    'Precision': precision,
    'Sensitivity': sensitivity,
    'F1 Score': f1,
    'MCC': mcc,
    'Markedness': markedness,
    "Youden's J": youdens_j,
    'FMI': fmi,
    'Time': total_time
}

performance = pd.concat([performance, pd.DataFrame([newRow])], ignore_index=True)
performance.to_csv("Model_Selection.csv", index=False,)
performance

In [ ]:
import pandas as pd
import numpy as np
import time
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input, BatchNormalization, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, roc_auc_score,
                             confusion_matrix)

# Define Sparse Neural Network block with Batch Normalization and Dropout
class SparseLayer(tf.keras.layers.Layer):
    def __init__(self, units, sparsity_coefficient=0.001, **kwargs):
        super(SparseLayer, self).__init__(**kwargs)
        self.units = units
        self.sparsity_coefficient = sparsity_coefficient

    def build(self, input_shape):
        # Define weights and biases with L1 regularization
        self.w = self.add_weight(shape=(input_shape[-1], self.units),
                                 initializer="random_normal",
                                 regularizer=tf.keras.regularizers.L1(self.sparsity_coefficient),
                                 trainable=True)
        self.b = self.add_weight(shape=(self.units,), initializer="zeros", trainable=True)

    def call(self, inputs):
        return tf.nn.relu(tf.matmul(inputs, self.w) + self.b)

# Define Sparse Neural Network Model with Batch Normalization
def build_sparse_nn(input_dim, sparsity_coefficient=0.001):
    inputs = Input(shape=(input_dim,))
    
    # Sparse layers with L1 regularization and Batch Normalization
    x = SparseLayer(256, sparsity_coefficient=sparsity_coefficient)(inputs)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    
    x = SparseLayer(128, sparsity_coefficient=sparsity_coefficient)(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    
    x = SparseLayer(64, sparsity_coefficient=sparsity_coefficient)(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    # Output layer
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=inputs, outputs=outputs)
    return model

# Initialize and compile the Sparse Neural Network model with adjusted learning rate
model = build_sparse_nn(input_dim=X_train.shape[1])
model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

# Learning rate scheduler: Reduce learning rate if validation loss stops improving
lr_reduction = ReduceLROnPlateau(monitor='val_loss', 
                                 patience=3, 
                                 verbose=1, 
                                 factor=0.5, 
                                 min_lr=1e-6)

# Early stopping to avoid overfitting
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Train model and time the training process (10 epochs)
start_time = time.time()
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2, 
                    verbose=1, callbacks=[lr_reduction, early_stopping])
end_time = time.time()

# Predictions and performance metrics
y_pred = (model.predict(X_test) > 0.5).astype("int32")

# Compute metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred)

# Additional metrics: Markedness, Youden's J statistic, and Fowlkes-Mallows Index
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
markedness = (precision * (tp / (tp + fn))) - ((1 - precision) * (fn / (fn + tp))) if (tp + fn) > 0 and (fn + tp) > 0 else np.nan
youdens_j = sensitivity + (tn / (tn + fp)) - 1
fmi = np.sqrt(precision * sensitivity)

# Calculate total time taken
total_time = end_time - start_time

# Create the performance dictionary
newRow = {
    'Name': 'Sparse Neural Network (SNN) with Batch Normalization',
    'Accuracy': accuracy,
    'Precision': precision,
    'Sensitivity': sensitivity,
    'F1 Score': f1,
    'MCC': mcc,
    'Markedness': markedness,
    "Youden's J": youdens_j,
    'FMI': fmi,
    'Time': total_time
}

performance = pd.concat([performance, pd.DataFrame([newRow])], ignore_index=True)
performance.to_csv("Model_Selection.csv", index=False, mode='a')
performance


In [ ]:
import pandas as pd
import numpy as np
import time
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input, Multiply, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, roc_auc_score,
                             confusion_matrix)

# Define Attention Layer
class AttentionLayer(tf.keras.layers.Layer):
    def __init__(self, units, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)
        self.units = units
        self.attention_dense = Dense(units, activation='tanh')
        self.attention_weights = Dense(1, activation='softmax')
    
    def call(self, inputs):
        # Compute attention scores
        attention_scores = self.attention_dense(inputs)
        attention_scores = self.attention_weights(attention_scores)
        # Apply attention scores to input features
        weighted_inputs = Multiply()([inputs, attention_scores])
        return weighted_inputs

# Define Attention-based MLP Model with reduced complexity
def build_attention_mlp(input_dim, attention_units=32, dense_units=[64, 32, 16]):
    inputs = Input(shape=(input_dim,))
    
    # Attention layer
    attention_layer = AttentionLayer(units=attention_units)
    attention_output = attention_layer(inputs)
    
    # Dense layers after attention with reduced units and added dropout
    x = Dense(dense_units[0], activation='relu')(attention_output)
    x = Dropout(0.4)(x)
    x = Dense(dense_units[1], activation='relu')(x)
    x = Dropout(0.4)(x)
    x = Dense(dense_units[2], activation='relu')(x)
    x = Dropout(0.4)(x)
    
    # Output layer
    outputs = Dense(1, activation='sigmoid')(x)
    
    model = Model(inputs=inputs, outputs=outputs)
    return model

# Initialize and compile the Attention-based MLP model with lower learning rate
model = build_attention_mlp(input_dim=X_train.shape[1])
model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

# Early stopping to avoid overfitting and ensure we stop training when performance stagnates
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Train model and time the training process (10 epochs max)
start_time = time.time()
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2, 
                    verbose=1, callbacks=[early_stopping])
end_time = time.time()

# Predictions and performance metrics
y_pred = (model.predict(X_test) > 0.5).astype("int32")

# Compute metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred)

# Additional metrics: Markedness, Youden's J statistic, and Fowlkes-Mallows Index
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
markedness = (precision * (tp / (tp + fn))) - ((1 - precision) * (fn / (fn + tp))) if (tp + fn) > 0 and (fn + tp) > 0 else np.nan
youdens_j = sensitivity + (tn / (tn + fp)) - 1
fmi = np.sqrt(precision * sensitivity)

# Calculate total time taken
total_time = end_time - start_time

# Create the performance dictionary
newRow = {
    'Name': 'Attention-based MLP Model (Adjusted)',
    'Accuracy': accuracy,
    'Precision': precision,
    'Sensitivity': sensitivity,
    'F1 Score': f1,
    'MCC': mcc,
    'Markedness': markedness,
    "Youden's J": youdens_j,
    'FMI': fmi,
    'Time': total_time
}

performance = pd.concat([performance, pd.DataFrame([newRow])], ignore_index=True)
performance.to_csv("Model_Selection.csv", index=False, mode='a')
performance


In [ ]:
import pandas as pd
import numpy as np
import time
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input, Layer, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, roc_auc_score,
                             confusion_matrix)

# Define NODE Block
class NODEBlock(Layer):
    def __init__(self, num_trees, tree_depth, output_dim, **kwargs):
        super(NODEBlock, self).__init__(**kwargs)
        self.num_trees = num_trees
        self.tree_depth = tree_depth
        self.output_dim = output_dim

        # Initialize layers for the "oblivious" decision trees
        self.feature_selectors = [Dense(1) for _ in range(tree_depth * num_trees)]
        self.tree_weights = Dense(output_dim)

    def call(self, inputs):
        # Compute tree outputs
        tree_outputs = []
        for tree in range(self.num_trees):
            tree_output = inputs
            for depth in range(self.tree_depth):
                feature_selector = self.feature_selectors[tree * self.tree_depth + depth]
                tree_output = tf.sigmoid(feature_selector(tree_output))
            tree_outputs.append(tree_output)

        # Concatenate and combine tree outputs
        combined_trees = tf.concat(tree_outputs, axis=1)
        node_output = self.tree_weights(combined_trees)
        return node_output

# Define the NODE-based model
def build_node_model(input_dim, num_trees=3, tree_depth=2, output_dim=32):
    inputs = Input(shape=(input_dim,))
    
    # NODE Block
    node_block = NODEBlock(num_trees=num_trees, tree_depth=tree_depth, output_dim=output_dim)
    node_output = node_block(inputs)
    
    # Additional dense layers for hybrid model with dropout for regularization
    x = Dense(64, activation='relu')(node_output)
    x = Dropout(0.4)(x)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.4)(x)
    outputs = Dense(1, activation='sigmoid')(x)
    
    model = Model(inputs=inputs, outputs=outputs)
    return model

# Initialize and compile the NODE-based model with lower learning rate
model = build_node_model(input_dim=X_train.shape[1])
model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

# Early stopping to avoid overfitting and ensure we stop training when performance stagnates
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Train model and time the training process (10 epochs max)
start_time = time.time()
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2, 
                    verbose=1, callbacks=[early_stopping])
end_time = time.time()

# Predictions and performance metrics
y_pred = (model.predict(X_test) > 0.5).astype("int32")

# Compute metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred)

# Additional metrics: Markedness, Youden's J statistic, and Fowlkes-Mallows Index
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
markedness = (precision * (tp / (tp + fn))) - ((1 - precision) * (fn / (fn + tp))) if (tp + fn) > 0 and (fn + tp) > 0 else np.nan
youdens_j = sensitivity + (tn / (tn + fp)) - 1
fmi = np.sqrt(precision * sensitivity)

# Calculate total time taken
total_time = end_time - start_time

# Create the performance dictionary
newRow = {
    'Name': 'NODE-based Model',
    'Accuracy': accuracy,
    'Precision': precision,
    'Sensitivity': sensitivity,
    'F1 Score': f1,
    'MCC': mcc,
    'Markedness': markedness,
    "Youden's J": youdens_j,
    'FMI': fmi,
    'Time': total_time
}

performance = pd.concat([performance, pd.DataFrame([newRow])], ignore_index=True)
performance.to_csv("Model_Selection.csv", index=False, mode='a')
performance


In [ ]:
import pandas as pd
import numpy as np
import time
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input, Conv2D, GlobalAveragePooling2D, Multiply, Reshape, BatchNormalization, ReLU, Flatten
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score, confusion_matrix

# Pad data to match the 32x32 input shape
X_train_padded = np.pad(X_train, ((0, 0), (0, max(0, 32 * 32 - X_train.shape[1]))), 'constant').reshape(-1, 32, 32, 1)
X_test_padded = np.pad(X_test, ((0, 0), (0, max(0, 32 * 32 - X_test.shape[1]))), 'constant').reshape(-1, 32, 32, 1)

# Define the Squeeze-and-Excitation (SE) block
def se_block(input_tensor, reduction_ratio=16):
    """Creates a Squeeze-and-Excitation block with a given reduction ratio."""
    channels = input_tensor.shape[-1]
    se = GlobalAveragePooling2D()(input_tensor)
    se = Reshape((1, 1, channels))(se)
    se = Dense(channels // reduction_ratio, activation='relu')(se)
    se = Dense(channels, activation='sigmoid')(se)
    x = Multiply()([input_tensor, se])
    return x

# Define a simple CNN with SE blocks
inputs = Input(shape=(32, 32, 1))

# Initial Conv Layer
x = Conv2D(32, kernel_size=3, strides=2, padding='same')(inputs)
x = BatchNormalization()(x)
x = ReLU()(x)
x = se_block(x)  # Add SE block

# Second Conv Layer
x = Conv2D(64, kernel_size=3, strides=2, padding='same')(x)
x = BatchNormalization()(x)
x = ReLU()(x)
x = se_block(x)  # Add SE block

# Third Conv Layer
x = Conv2D(128, kernel_size=3, strides=2, padding='same')(x)
x = BatchNormalization()(x)
x = ReLU()(x)
x = se_block(x)  # Add SE block

# Global Average Pooling and output layers
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dense(64, activation='relu')(x)
outputs = Dense(1, activation='sigmoid')(x)

# Build the model
model = Model(inputs, outputs)

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train model and time the training process
start_time = time.time()
history = model.fit(X_train_padded, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=1)
end_time = time.time()

# Predictions and performance metrics
y_pred = (model.predict(X_test_padded) > 0.5).astype("int32")

# Compute metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred)

# Additional metrics: Markedness, Youden's J statistic, and Fowlkes-Mallows Index
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
markedness = (precision * (tp / (tp + fn))) - ((1 - precision) * (fn / (fn + tp))) if (tp + fn) > 0 and (fn + tp) > 0 else np.nan
youdens_j = sensitivity + (tn / (tn + fp)) - 1
fmi = np.sqrt(precision * sensitivity)  # Fowlkes-Mallows Index

# Calculate total time taken
total_time = end_time - start_time

# Create the performance dictionary
newRow = {
    'Name': 'SENet Hybrid Model',
    'Accuracy': accuracy,
    'Precision': precision,
    'Sensitivity': sensitivity,
    'F1 Score': f1,
    'MCC': mcc,
    'Markedness': markedness,
    "Youden's J": youdens_j,
    'FMI': fmi,
    'Time': total_time
}

performance = pd.concat([performance, pd.DataFrame([newRow])], ignore_index=True)
performance.to_csv("Model_Selection.csv", index=False, mode='a')
performance

In [ ]:
import pandas as pd
import numpy as np
import time
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input, Conv2D, DepthwiseConv2D, BatchNormalization, ReLU, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import LearningRateScheduler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score, confusion_matrix

# Pad data to match the 32x32 input shape
X_train_padded = np.pad(X_train, ((0, 0), (0, max(0, 32 * 32 - X_train.shape[1]))), 'constant').reshape(-1, 32, 32, 1)
X_test_padded = np.pad(X_test, ((0, 0), (0, max(0, 32 * 32 - X_test.shape[1]))), 'constant').reshape(-1, 32, 32, 1)

def micronet_block(x, out_channels, stride=1):
    x = DepthwiseConv2D(kernel_size=3, strides=stride, padding='same', depth_multiplier=1)(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = Conv2D(out_channels, kernel_size=1, strides=1, padding='same')(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    return x

# Model definition
inputs = Input(shape=(32, 32, 1))
x = Conv2D(16, kernel_size=3, strides=2, padding='same')(inputs)
x = BatchNormalization()(x)
x = ReLU()(x)
x = micronet_block(x, 32, stride=2)
x = micronet_block(x, 64, stride=2)
x = micronet_block(x, 128, stride=2)
x = GlobalAveragePooling2D()(x)
x = Dense(64, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01))(x)  # Weight decay added
x = Dropout(0.5)(x)  # Increased dropout
x = Dense(32, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01))(x)  # Reduced neurons
x = Dropout(0.5)(x)  # Increased dropout
outputs = Dense(1, activation='sigmoid')(x)

model = Model(inputs, outputs)

# Compile with weight decay
model.compile(optimizer=Adam(learning_rate=0.01), loss='binary_crossentropy', metrics=['accuracy'])

# Learning rate schedule with faster decay
def lr_schedule(epoch):
    return 0.01 * (0.4 ** (epoch // 3))  # Increased decay rate

callbacks = [LearningRateScheduler(lr_schedule)]

# Train the model
start_time = time.time()
history = model.fit(X_train_padded, y_train, epochs=10, batch_size=16, validation_split=0.2, verbose=1, callbacks=callbacks)
end_time = time.time()

# Predictions
y_pred = (model.predict(X_test_padded) > 0.5).astype("int32")

# Compute metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred)

# Additional metrics
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
markedness = (precision * (tp / (tp + fn))) - ((1 - precision) * (fn / (fn + tp))) if (tp + fn) > 0 and (fn + tp) > 0 else np.nan
youdens_j = sensitivity + (tn / (tn + fp)) - 1
fmi = np.sqrt(precision * sensitivity)

# Total time taken
total_time = end_time - start_time

# Performance dictionary
newRow = {
    'Name': 'MicroNet Hybrid Model',
    'Accuracy': accuracy,
    'Precision': precision,
    'Sensitivity': sensitivity,
    'F1 Score': f1,
    'MCC': mcc,
    'Markedness': markedness,
    "Youden's J": youdens_j,
    'FMI': fmi,
    'Time': total_time
}

performance = pd.concat([performance, pd.DataFrame([newRow])], ignore_index=True)
performance.to_csv("Model_Selection.csv", index=False, mode='a')
performance

In [ ]:
import pandas as pd
import numpy as np
import time
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input, BatchNormalization, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, roc_auc_score,
                             confusion_matrix)

def build_ffnn(input_dim):
    inputs = Input(shape=(input_dim,))
    
    # Improved architecture with Batch Normalization and Dropout
    x = Dense(128, activation='relu', kernel_initializer='he_normal')(inputs)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    
    x = Dense(64, activation='relu', kernel_initializer='he_normal')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    
    x = Dense(32, activation='relu', kernel_initializer='he_normal')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    
    # Output layer
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=inputs, outputs=outputs)
    return model

# Initialize and compile the Feed-Forward Neural Network model
model = build_ffnn(input_dim=X_train.shape[1])
model.compile(optimizer=Adam(learning_rate=0.005), loss='binary_crossentropy', metrics=['accuracy'])

# Train model and time the training process
start_time = time.time()
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=1)
end_time = time.time()

# Predictions and performance metrics
y_pred = (model.predict(X_test) > 0.5).astype("int32")

# Compute metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred)

# Additional metrics: Markedness, Youden's J statistic, and Fowlkes-Mallows Index
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
markedness = (precision * (tp / (tp + fn))) - ((1 - precision) * (fn / (fn + tp))) if (tp + fn) > 0 and (fn + tp) > 0 else np.nan
youdens_j = sensitivity + (tn / (tn + fp)) - 1
fmi = np.sqrt(precision * sensitivity)

# Calculate total time taken
total_time = end_time - start_time

# Create the performance dictionary
newRow = {
    'Name': 'Feed-Forward Neural Network',
    'Accuracy': accuracy,
    'Precision': precision,
    'Sensitivity': sensitivity,
    'F1 Score': f1,
    'MCC': mcc,
    'Markedness': markedness,
    "Youden's J": youdens_j,
    'FMI': fmi,
    'Time': total_time
}

performance = pd.concat([performance, pd.DataFrame([newRow])], ignore_index=True)
performance.to_csv("Model_Selection.csv", index=False, mode='a')
performance

In [ ]:
import pandas as pd
import numpy as np
import time
from sklearn.neural_network import BernoulliRBM
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score, confusion_matrix

# Assuming X_train, X_test, y_train, and y_test are defined
# Normalize data to [0, 1] range for RBM
scaler = MinMaxScaler()
X_train_normalized = scaler.fit_transform(X_train)
X_test_normalized = scaler.transform(X_test)

# Define the RBM model
rbm = BernoulliRBM(n_components=64, learning_rate=0.01, n_iter=10, random_state=42)
classifier = Pipeline(steps=[('rbm', rbm), ('logistic', None)])

# Train model and time the training process
start_time = time.time()
rbm.fit(X_train_normalized)
end_time = time.time()

# Transform features using RBM
X_train_rbm = rbm.transform(X_train_normalized)
X_test_rbm = rbm.transform(X_test_normalized)

# Use logistic regression as a classifier
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(X_train_rbm, y_train)
y_pred = clf.predict(X_test_rbm)

# Compute metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred)

# Additional metrics: Markedness, Youden's J statistic, and Fowlkes-Mallows Index
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
markedness = (precision * (tp / (tp + fn))) - ((1 - precision) * (fn / (fn + tp))) if (tp + fn) > 0 and (fn + tp) > 0 else np.nan
youdens_j = sensitivity + (tn / (tn + fp)) - 1
fmi = np.sqrt(precision * sensitivity)  # Fowlkes-Mallows Index

# Calculate total time taken
total_time = end_time - start_time

# Create the performance dictionary
newRow = {
    'Name': 'Restricted Boltzmann Machine Model',
    'Accuracy': accuracy,
    'Precision': precision,
    'Sensitivity': sensitivity,
    'F1 Score': f1,
    'MCC': mcc,
    'Markedness': markedness,
    "Youden's J": youdens_j,
    'FMI': fmi,
    'Time': total_time
}

performance = pd.concat([performance, pd.DataFrame([newRow])], ignore_index=True)
performance.to_csv("Model_Selection.csv", index=False, mode='a')
performance

In [ ]:
import pandas as pd
import numpy as np
import time
import tensorflow as tf
from tensorflow.keras.layers import Dense, Flatten, Input, Conv2D, BatchNormalization, ReLU, Add, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score, confusion_matrix

# Load and preprocess data (example placeholders)
# Replace these with your actual data loading process
# X_train, X_test, y_train, y_test = ...

# Pad data to match the 32x32 input shape
X_train_padded = np.pad(X_train, ((0, 0), (0, max(0, 32 * 32 - X_train.shape[1]))), 'constant').reshape(-1, 32, 32, 1)
X_test_padded = np.pad(X_test, ((0, 0), (0, max(0, 32 * 32 - X_test.shape[1]))), 'constant').reshape(-1, 32, 32, 1)

# Define a RegNet block
def regnet_block(x, out_channels, stride=1):
    """RegNet Block with a bottleneck structure."""
    shortcut = x

    # 1x1 Convolution (Bottleneck Layer)
    x = Conv2D(out_channels, kernel_size=1, strides=1, padding='same')(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    # 3x3 Convolution (Main Convolution)
    x = Conv2D(out_channels, kernel_size=3, strides=stride, padding='same')(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    # 1x1 Convolution (Expansion Layer)
    x = Conv2D(out_channels, kernel_size=1, strides=1, padding='same')(x)
    x = BatchNormalization()(x)

    # Shortcut connection with a downsample layer if needed
    if stride != 1 or shortcut.shape[-1] != out_channels:
        shortcut = Conv2D(out_channels, kernel_size=1, strides=stride, padding='same')(shortcut)
        shortcut = BatchNormalization()(shortcut)

    # Add shortcut and main path
    x = Add()([x, shortcut])
    x = ReLU()(x)
    return x

# Define the RegNet-inspired backbone model
inputs = Input(shape=(32, 32, 1))
x = Conv2D(32, kernel_size=3, strides=2, padding='same')(inputs)
x = BatchNormalization()(x)
x = ReLU()(x)

# Stack several RegNet blocks
x = regnet_block(x, 64, stride=2)
x = regnet_block(x, 128, stride=2)
x = regnet_block(x, 256, stride=2)

x = GlobalAveragePooling2D()(x)

base_model = Model(inputs, x)

# Build the final model
x = Dense(128, activation='relu')(base_model.output)
x = Dense(64, activation='relu')(x)
outputs = Dense(1, activation='sigmoid')(x)
model = Model(inputs, outputs)

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train model and time the training process
start_time = time.time()
history = model.fit(X_train_padded, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=1)
end_time = time.time()

# Predictions and performance metrics
y_pred = (model.predict(X_test_padded) > 0.5).astype("int32")

# Compute metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred)

# Additional metrics: Markedness, Youden's J statistic, and Fowlkes-Mallows Index
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
markedness = (precision * (tp / (tp + fn))) - ((1 - precision) * (fn / (fn + tp))) if (tp + fn) > 0 and (fn + tp) > 0 else np.nan
youdens_j = sensitivity + (tn / (tn + fp)) - 1
fmi = np.sqrt(precision * sensitivity)  # Fowlkes-Mallows Index

# Calculate total time taken
total_time = end_time - start_time

# Create the performance dictionary
newRow = {
    'Name': 'RegNet Hybrid Model',
    'Accuracy': accuracy,
    'Precision': precision,
    'Sensitivity': sensitivity,
    'F1 Score': f1,
    'MCC': mcc,
    'Markedness': markedness,
    "Youden's J": youdens_j,
    'FMI': fmi,
    'Time': total_time
}

performance = pd.concat([performance, pd.DataFrame([newRow])], ignore_index=True)
performance.to_csv("Model_Selection.csv", index=False, mode='a')
performance

In [ ]:
import pandas as pd
import numpy as np
import time
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input, Dropout, LayerNormalization, MultiHeadAttention, Add, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# Address class imbalance using SMOTE
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Normalize the data
scaler = StandardScaler()
X_train_resampled = scaler.fit_transform(X_train_resampled)
X_test = scaler.transform(X_test)

# Reshape data to match the required input format for Transformers: (samples, timesteps, features)
X_train_reshaped = X_train_resampled.reshape(X_train_resampled.shape[0], X_train_resampled.shape[1], 1)
X_test_reshaped = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

# Define Transformer Encoder block
def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0.1):
    # Multi-Head Attention
    attention = MultiHeadAttention(num_heads=num_heads, key_dim=head_size, dropout=dropout)(inputs, inputs)
    attention = Dropout(dropout)(attention)
    attention = Add()([attention, inputs])
    attention = LayerNormalization(epsilon=1e-6)(attention)

    # Feed-Forward Network
    ff = Dense(ff_dim, activation="relu")(attention)
    ff = Dense(inputs.shape[-1])(ff)
    ff = Dropout(dropout)(ff)
    ff = Add()([ff, attention])
    ff = LayerNormalization(epsilon=1e-6)(ff)

    return ff

# Define the Transformer model
inputs = Input(shape=(X_train_reshaped.shape[1], 1))
x = transformer_encoder(inputs, head_size=64, num_heads=4, ff_dim=128, dropout=0.2)
x = transformer_encoder(x, head_size=64, num_heads=4, ff_dim=128, dropout=0.2)
x = transformer_encoder(x, head_size=64, num_heads=4, ff_dim=128, dropout=0.2)  # Added an additional layer
x = GlobalAveragePooling1D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.4)(x)  # Increased dropout
x = Dense(64, activation='relu')(x)
x = Dropout(0.4)(x)
outputs = Dense(1, activation='sigmoid')(x)
model = Model(inputs, outputs)

# Compile the model with a learning rate scheduler
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=0.001, decay_steps=1000, decay_rate=0.9
)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Train model and time the training process
start_time = time.time()
history = model.fit(X_train_reshaped, y_train_resampled, epochs=10, batch_size=32, validation_split=0.2, verbose=1)
end_time = time.time()

# Predictions and performance metrics
y_pred_prob = model.predict(X_test_reshaped)
y_pred = (y_pred_prob > 0.5).astype("int32")

# Compute metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_prob)

# Additional metrics: Markedness, Youden's J statistic, and Fowlkes-Mallows Index
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
markedness = (precision * (tp / (tp + fn))) - ((1 - precision) * (fn / (fn + tp))) if (tp + fn) > 0 and (fn + tp) > 0 else np.nan
youdens_j = sensitivity + (tn / (tn + fp)) - 1
fmi = np.sqrt(precision * sensitivity)  # Fowlkes-Mallows Index

# Calculate total time taken
total_time = end_time - start_time

# Create the performance dictionary
newRow = {
    'Name': 'Improved Transformer Model with SMOTE',
    'Accuracy': accuracy,
    'Precision': precision,
    'Sensitivity': sensitivity,
    'F1 Score': f1,
    'MCC': mcc,
    'ROC AUC': roc_auc,
    'Markedness': markedness,
    "Youden's J": youdens_j,
    'FMI': fmi,
    'Time': total_time
}

performance = pd.concat([performance, pd.DataFrame([newRow])], ignore_index=True)
performance.to_csv("Model_Selection.csv", index=False, mode='a')
performance


In [ ]:
performance = pd.read_csv("Model_Selection.csv")
performance

In [ ]:
performance=performance.head(9)
performance